# Homework 4, Problem 5: Training a simple neural network

This notebook uses only PyTorch, NumPy, and Matplotlib. No GPU or dataset
download is needed. In VS Code, select a Python kernel with these packages
installed; Jupyter and Colab's CPU runtime also work.

In [1]:
from pathlib import Path
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

BASE_SEED = 67900
DEVICE = torch.device("cpu")
# Small matrix operations usually run faster without many CPU threads.
torch.set_num_threads(1)
CHECKPOINT_ROOT = Path("checkpoints") / "hw4_nn"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})
CLASS_COLORS = plt.get_cmap("Set1")(np.arange(3))[:, :3]
CLASS_CMAP = ListedColormap(CLASS_COLORS)
figures = {}
print("CPU runtime; separate fixed seeds for data, initialization, and batches.")

CPU runtime; separate fixed seeds for data, initialization, and batches.


In [ ]:
class DataDistribution:
    def __init__(self, rng, centers_radius, variance, num_classes):
        if centers_radius <= 0 or variance <= 0 or num_classes < 2:
            raise ValueError("Use a positive radius/variance and at least two classes.")
        self.rng = rng
        self.variance = float(variance)
        self.num_classes = int(num_classes)
        self.dim = 2
        angles = 2 * torch.pi * torch.arange(num_classes) / num_classes
        self.centers = centers_radius * torch.stack(
            (torch.cos(angles), torch.sin(angles)), dim=1)

    def sample(self, n_samples, generator=None):
        generator = self.rng if generator is None else generator
        y = torch.randint(self.num_classes, (n_samples,), generator=generator)
        noise = torch.randn(n_samples, self.dim, generator=generator)
        x = self.centers[y] + self.variance**0.5 * noise
        return x, y

dist = DataDistribution(torch.Generator().manual_seed(BASE_SEED), 1.0, 0.5, 3)

def make_fixed_data():
    data_generator = torch.Generator().manual_seed(BASE_SEED + 2)
    return {
        "train": dist.sample(250, generator=data_generator),
        "validation": dist.sample(500, generator=data_generator),
        "test": dist.sample(2000, generator=data_generator),
    }

fixed_data = make_fixed_data()
X_train, Y_train = fixed_data["train"]
X_valid, Y_valid = fixed_data["validation"]
X_test, Y_test = fixed_data["test"]
ITERATIONS = 10000
LEARNING_RATE = 0.01
BATCH_SIZE = 100
CHECKPOINT_EVERY = 50
MODEL_SEED = BASE_SEED + 3
BATCH_SEED = BASE_SEED + 4
BASELINE_CONFIG = {"depth": 2, "width": 10}

@torch.no_grad()
def dataset_metrics(model, x, y):
    logits = model(x)
    return {
        "loss": torch.nn.functional.cross_entropy(logits, y).item(),
        "error": (logits.argmax(dim=1) != y).float().mean().item(),
    }

print(f"Training budget: {ITERATIONS:,} SGD updates per network.")

Training budget: 10,000 SGD updates per network.


: 

## (a) The data distribution

The data-generating process is as follows. Labels are sampled uniformly at random, and conditional on each label, the covariates are drawn from a Gaussian distribution centered at the corresponding class center.

\begin{align*}
    y &\sim \text{Uniform}(\{0, 1, 2\})\\
    \boldsymbol{x} \mid y = c &\sim \mathcal{N}(\boldsymbol{\mu}_c, 0.5 \mathbf{I}_2)\\
    \boldsymbol{\mu}_c &= \begin{pmatrix} \cos(2\pi c/3)\\\sin(2\pi c/3)\end{pmatrix},
\qquad c\in\{0,1,2\}.
\end{align*}

In [ ]:
plot_generator = torch.Generator().manual_seed(BASE_SEED + 1)
X_plot_samples, Y_plot_samples = dist.sample(1000, generator=plot_generator)

def add_reference_boundaries(ax):
    angles = np.pi / 3 + 2 * np.pi * np.arange(3) / 3
    for angle in angles:
        ax.plot([0, 6*np.cos(angle)], [0, 6*np.sin(angle)], "k--", lw=1.2)

def plot_distribution(show_boundaries=False):
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    for c in range(3):
        mask = Y_plot_samples.numpy() == c
        points = X_plot_samples.numpy()[mask]
        ax.scatter(points[:, 0], points[:, 1], s=12, alpha=0.55,
                   color=CLASS_COLORS[c], label=f"Class {c}")

    ax.scatter(*dist.centers.numpy().T, marker="x", color="black", s=75,
               label="Class centers")

    add_reference_boundaries(ax)
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Three-class Gaussian mixture",
           xlim=(-3, 3), ylim=(-3, 3))
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    return fig

figures["distribution"] = plot_distribution(show_boundaries=True)
plt.show()

## (b) Implement the network

Implement the functions `Neuron.forward()`, `Layer.__init__()`, and `FullyConnectedNetwork.__init__()` below.

In [ ]:
class Neuron(nn.Module):
    def __init__(self, in_features, activation, generator=None):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, 1, generator=generator))
        self.bias = nn.Parameter(torch.randn(1, generator=generator))
        self.activation = activation

    def forward(self, x):
        # TODO: compute the affine preactivation.
        z = x * self.weight + self.bias                      
        return self.activation(z)

class Layer(nn.Module):
    def __init__(self, in_features, out_features, activation, generator=None):
        super().__init__()

        # TODO: initialize out_features neurons.
        neuron_list = []
        for _ in range(out_features):
            neuron = Neuron(
                in_features = in_features, 
                activation = activation,
                generator = generator
            )
            neuron_list.append(neuron)
        self.neurons = nn.ModuleList(neuron_list)

    def forward(self, x):
        return torch.cat([neuron(x) for neuron in self.neurons], dim=1)

class FullyConnectedNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, depth, generator=None):
        super().__init__()
        if depth < 1 or hidden_dim < 1:
            raise ValueError("Depth and width must be positive integers.")

        # TODO: hidden ReLU layers followed by a linear output.
        layers = []
        for _ in range(depth):
            layers = Layer(
                in_features = input_dim,
                out_features = output_dim,
                activation = nn.ReLU,
                generator = generator
            )
        self.layers = nn.ModuleList(layers)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def loss(self, x, y):
        return self.loss_fn(self(x), y)

    def inference_probs(self, x):
        return torch.softmax(self(x), dim=1)

    def inference_classes(self, x):
        return self(x).argmax(dim=1)

In [ ]:
# Quick checks for part (b).
with torch.random.fork_rng(devices=[]):
    for depth, width in [(1, 4), (2, 10), (4, 10)]:
        model = FullyConnectedNetwork(2, width, 3, depth,
                                      torch.Generator().manual_seed(123))
        x = torch.tensor([[0.2, -0.4], [1.0, 0.5], [-0.3, 0.8]])
        y = torch.tensor([0, 1, 2])
        assert model(x).shape == (3, 3), "Expected one logit per example/class."
        probs = model.inference_probs(x)
        assert torch.allclose(probs.sum(dim=1), torch.ones(3), atol=1e-6)
        loss = model.loss(x, y)
        assert loss.ndim == 0 and torch.isfinite(loss)
        loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all()
                   for p in model.parameters()), "A parameter is disconnected from the loss."
print("Part (b): shapes, probabilities, and gradient connectivity passed.")

## (c), (d) Train the baseline network

The notebook trains for the full budget and then restores the saved
checkpoint with minimum validation cross-entropy. It does not terminate
training at the first increase in validation loss.

In [ ]:
def train_network(config, name, model_seed=MODEL_SEED, batch_seed=BATCH_SEED,
                  iterations=None, checkpoint_root=CHECKPOINT_ROOT,
                  verbose=True):
    if iterations is None:
        iterations = ITERATIONS
    init_generator = torch.Generator().manual_seed(model_seed)
    batch_generator = torch.Generator().manual_seed(batch_seed)
    model = FullyConnectedNetwork(2, config["width"], 3, config["depth"],
                                  init_generator)
    model.train()
    directory = Path(checkpoint_root) / (
        f"{name}_d{config['depth']}_w{config['width']}_seed_{model_seed}")
    directory.mkdir(parents=True, exist_ok=True)
    history = []
    best_loss = float("inf")
    best_step = None
    started = time.perf_counter()

    def record_checkpoint(step):
        nonlocal best_loss, best_step
        train = dataset_metrics(model, X_train, Y_train)
        valid = dataset_metrics(model, X_valid, Y_valid)
        row = {"step": step, "train_loss": train["loss"],
               "validation_loss": valid["loss"], "train_error": train["error"],
               "validation_error": valid["error"]}
        if not all(np.isfinite(value) for value in row.values()):
            raise RuntimeError("Non-finite training metric; check the implementation.")
        history.append(row)
        torch.save(model.state_dict(), directory / f"step_{step:04d}.pt")
        if valid["loss"] < best_loss:
            best_loss, best_step = valid["loss"], step
            torch.save(model.state_dict(), directory / "best.pt")
        if verbose and (step % 1000 == 0 or step == iterations):
            print(f"{name}, checkpoint {step:4d}: train CE={train['loss']:.4f}, "
                  f"validation CE={valid['loss']:.4f}")

    record_checkpoint(0)
    for step in range(1, iterations + 1):
        indices = torch.randperm(len(X_train), generator=batch_generator)[:BATCH_SIZE]
        loss = model.loss(X_train[indices], Y_train[indices])
        loss.backward()
        with torch.no_grad():
            for parameter in model.parameters():
                parameter -= LEARNING_RATE * parameter.grad

        # Part (c): PyTorch accumulates gradients unless we clear them.
        model.zero_grad(set_to_none=True)
        if step % CHECKPOINT_EVERY == 0 or step == iterations:
            record_checkpoint(step)

    model.eval()
    torch.save(model.state_dict(), directory / "final.pt")
    selected = FullyConnectedNetwork(2, config["width"], 3, config["depth"],
                                     torch.Generator().manual_seed(model_seed))
    selected.load_state_dict(torch.load(directory / "best.pt", map_location="cpu",
                                        weights_only=True))
    selected.eval()
    return {"name": name, "config": dict(config), "final_model": model,
            "selected_model": selected, "history": history, "best_step": best_step,
            "checkpoint_dir": directory,
            "seconds": time.perf_counter() - started}

baseline = train_network(BASELINE_CONFIG, "baseline")

In [ ]:
def plot_learning_curves(runs, title):
    fig, axes = plt.subplots(1, 2, figsize=(7.8, 4.2))
    colors = ["tab:blue", "tab:orange", "tab:green"]
    for run, color in zip(runs, colors):
        h = run["history"]
        steps = [row["step"] for row in h]
        label = f"Depth {run['config']['depth']}, width {run['config']['width']}"
        for ax, metric in zip(axes, ("loss", "error")):
            ax.plot(steps, [row[f"train_{metric}"] for row in h], color=color,
                    label=f"{label}: train")
            ax.plot(steps, [row[f"validation_{metric}"] for row in h], "--", color=color,
                    label=f"{label}: validation")
            ax.axvline(run["best_step"], color=color, alpha=0.3, lw=1)
    axes[0].set(ylabel="Mean cross-entropy", yscale="log")
    axes[1].set(ylabel="Classification error", ylim=(0, 1))
    for ax in axes:
        ax.set_xlabel("SGD updates")
        ax.grid(alpha=0.25)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9.5)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0.13, 1, 0.93))
    return fig

figures["baseline_learning"] = plot_learning_curves(
    [baseline], "Baseline (vertical line = validation-selected checkpoint)")
plt.show()

def plot_later_losses(runs, start_step=1000):
    # Include an earlier selected checkpoint if an alternative model needs it.
    start_step = min(start_step, min(run["best_step"] for run in runs))
    fig, axes = plt.subplots(1, len(runs), figsize=(8.2, 3.6),
                             squeeze=False, sharey=True)
    for ax, run in zip(axes[0], runs):
        history = [row for row in run["history"] if row["step"] >= start_step]
        steps = [row["step"] for row in history]
        ax.plot(steps, [row["train_loss"] for row in history],
                color="tab:blue", label="Training")
        ax.plot(steps, [row["validation_loss"] for row in history],
                color="tab:orange", label="Validation")
        selected = min(run["history"], key=lambda row: row["validation_loss"])
        ax.scatter(selected["step"], selected["validation_loss"], marker="*",
                   s=110, color="tab:orange", edgecolors="black", zorder=5,
                   label=f"Selected: {selected['step']:,}")
        ax.set(title=f"Depth {run['config']['depth']}, width {run['config']['width']}",
               xlabel="SGD updates")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)
    axes[0, 0].set_ylabel("Mean cross-entropy")
    fig.suptitle(f"Later training: updates {start_step:,} onward", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return fig


In [ ]:
grid_axis = torch.linspace(-3, 3, 120)
grid_x, grid_y = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
grid_inputs = torch.stack((grid_x.reshape(-1), grid_y.reshape(-1)), dim=1)

@torch.no_grad()
def grid_predictions(model):
    probabilities = model.inference_probs(grid_inputs)
    labels = probabilities.argmax(dim=1).reshape(grid_x.shape).numpy()
    confidence = probabilities.max(dim=1).values.reshape(grid_x.shape).numpy()
    return labels, confidence

def draw_regions(ax, model, title):
    labels, _ = grid_predictions(model)
    ax.contourf(grid_x.numpy(), grid_y.numpy(), labels,
                levels=np.arange(-0.5, 3.5), cmap=CLASS_CMAP, alpha=0.5)
    for c in range(3):
        mask = Y_train.numpy() == c
        points = X_train.numpy()[mask]
        ax.scatter(points[:, 0], points[:, 1], s=14, color=CLASS_COLORS[c],
                   edgecolors="black", linewidths=0.25, alpha=0.65)
    add_reference_boundaries(ax)
    ax.set(title=title, xlabel="$x_1$", ylabel="$x_2$", xlim=(-3, 3), ylim=(-3, 3))
    ax.set_aspect("equal")

def plot_regions_and_confidence(run):
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 4.0))
    draw_regions(axes[0], run["final_model"], "Final predicted classes")
    handles = [Line2D([], [], marker="o", linestyle="none", color=CLASS_COLORS[c],
                      label=f"Class {c}") for c in range(3)]
    handles.append(Line2D([], [], color="black", linestyle="--", label="Optimal boundary"))
    axes[0].legend(handles=handles, fontsize=9, loc="upper right")
    _, confidence = grid_predictions(run["final_model"])
    im = axes[1].imshow(confidence, extent=(-3, 3, -3, 3), origin="lower",
                        vmin=1/3, vmax=1, cmap="viridis", interpolation="nearest")
    add_reference_boundaries(axes[1])
    axes[1].set(title="Maximum class probability", xlabel="$x_1$", ylabel="$x_2$",
                xlim=(-3, 3), ylim=(-3, 3))
    axes[1].set_aspect("equal")
    fig.colorbar(im, ax=axes[1], label="Max softmax probability", fraction=0.047, pad=0.04)
    fig.tight_layout()
    return fig

figures["baseline_regions"] = plot_regions_and_confidence(baseline)
plt.show()

## (e) Increase depth or width and compare

Modestly increase the depth and/or width of the network from the earlier depth of 2 and width of 10. Do not change the datasets, batch sequence, number of updates, or learning rate.

In [ ]:
# TODO: Change the depth and/or width. The architecture trained in part (d)
# uses a depth of 2 and a width of 10.
COMPARISON_CONFIG = {"depth": ..., "width": ...}

comparison = train_network(COMPARISON_CONFIG, "changed_architecture")
runs = [baseline, comparison]
figures["comparison_learning"] = plot_learning_curves(
    runs, "Architecture comparison (fixed data and mini-batch order)")
plt.show()

figures["comparison_late_learning"] = plot_later_losses(runs)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8.0, 4.2))
for ax, run in zip(axes, runs):
    cfg = run["config"]
    draw_regions(ax, run["final_model"], f"Depth {cfg['depth']}, width {cfg['width']}")
fig.suptitle("Final decision regions (dashed = optimal boundaries)")
fig.tight_layout()
figures["comparison_regions"] = fig
plt.show()

## (f) Final evaluation

In [ ]:
summary = []
for run in runs:
    for checkpoint, model, step in (
        ("final", run["final_model"], ITERATIONS),
        ("min validation", run["selected_model"], run["best_step"]),
    ):
        if run["config"]["depth"] == 4 and checkpoint == "final":
            continue

        train = dataset_metrics(model, X_train, Y_train)
        valid = dataset_metrics(model, X_valid, Y_valid)
        test = dataset_metrics(model, X_test, Y_test)
        summary.append({"name": run["name"], "depth": run["config"]["depth"],
                        "width": run["config"]["width"], "checkpoint": checkpoint,
                        "step": step, "train_loss": train["loss"],
                        "validation_loss": valid["loss"], "test_loss": test["loss"],
                        "test_error": test["error"],
                        "gap": valid["loss"] - train["loss"]})

print(f"{'depth':<6} {'width':<6} {'selection':<14} {'index':<7} "
      f"{'train CE':<9} {'valid CE':<9} {'test CE':<9} {'test error':<11} {'gap':<8}")
for row in summary:
    print(f"{row['depth']:<6d} {row['width']:<6d} {row['checkpoint']:<14s} {row['step']:<7d} "
          f"{row['train_loss']:<9.4f} {row['validation_loss']:<9.4f} "
          f"{row['test_loss']:<9.4f} {row['test_error']:<11.4f} {row['gap']:<8.4f}")